# Notebook 2: 3D Gaussian Splatting sobre vistas SPC

**Proyecto INF449: SPC + 3D Gaussian Splatting.** Este notebook cubre los últimos dos pasos:

3. **Preparar** un directorio de escena válido por cada configuración SPC, con validaciones.
4. **Entrenar y comparar** 3DGS: imágenes originales (baseline) frente a cada configuración SPC.

Requiere haber ejecutado [el Notebook 1](01_spc_simulacion_reconstruccion.ipynb). El entrenamiento usa el
repo oficial sin modificar, mediante `codes/gs_train_runner.py`.

**Limitación que condiciona la lectura de los resultados.** Todas las corridas usan las poses y la nube SfM
calculadas por COLMAP sobre las imágenes originales. Una SPC real no dispondría de ellas: tendría que
estimarlas desde las vistas reconstruidas, o medirlas de otra forma. Este experimento aísla entonces el
efecto de la **calidad fotométrica**, y sus resultados son una **cota optimista** del sistema completo. El
paper muestra además que partir de la nube SfM mejora la calidad frente a una inicialización aleatoria
(§7.3, Fig. 7).

**Ejecución.** Las corridas van en secuencia. Cada una guarda sus resultados al terminar y, si algo se
interrumpe, volver a ejecutar el notebook omite lo terminado y retoma desde el último checkpoint.

## 1. Configuración

In [ ]:
DATASET = "tandt/truck"
SPC_RUN = "completa"           # carpeta creada por el Notebook 1
GS_RUN = "completa_r1_s0"      # subcarpeta en outputs/gs_runs/<escena>/

ITER_LONG = 30_000
ITER_SHORT = 7_000
HADAMARD = "hadamard_cc"       # orden Hadamard usado en las corridas principales

# PLAN = None usa el plan por defecto; una lista [[nombre, iteraciones], ...] lo reemplaza.
PLAN = None
ONLY = None                    # lista de nombres a ejecutar ahora; None = todo el plan

# Hiperparámetros: los valores del paper. Lo que se cambie aquí aplica a TODAS las corridas.
RESOLUTION = 1                 # 1 = tamaño del archivo (979x546 en Truck)
DATA_DEVICE = "cuda"           # "cpu" reduce VRAM a costa de velocidad
DENSIFY_GRAD_THRESHOLD = 0.0002
DENSIFY_UNTIL_ITER = 15_000
SEED = 0                       # la semilla que fija safe_state en el repo oficial

# Cuidado térmico del laptop: entre corridas se espera a que la GPU baje de esta temperatura
MAX_START_TEMP_C = 70
COOLDOWN_MAX_S = 600

STOP_ON_OOM = True             # sin VRAM, detenerse: cambiar parámetros exige repetir todas las corridas
EXPORT_GIF = True
LLFF_HOLD = 8

## 2. Entorno

In [ ]:
import csv
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import time
from argparse import ArgumentParser
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageDraw


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "third_party" / "gaussian-splatting").is_dir() and (path / "data").is_dir():
            return path
    raise FileNotFoundError(f"No encontré la raíz del repositorio desde {start}.")


ROOT = find_repo_root(Path.cwd().resolve())
GS_REPO = ROOT / "third_party" / "gaussian-splatting"
RUNNER = ROOT / "codes" / "gs_train_runner.py"
sys.path.insert(0, str(GS_REPO))

print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA de torch", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No hay CUDA. Selecciona el kernel de .venv, que tiene torch 2.11.0+cu128.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 2**30:.2f} GiB")

try:
    import diff_gaussian_rasterization
    import simple_knn._C
except ImportError as error:
    raise RuntimeError(
        "Faltan las extensiones CUDA de 3DGS. Compílalas siguiendo codes/README.md."
    ) from error

from scene.colmap_loader import read_extrinsics_binary, read_intrinsics_binary
from utils.graphics_utils import focal2fov

SCENE_DIR = ROOT / "data" / "tandt_db" / DATASET
SCENE_NAME = Path(DATASET).name
SPC_DIR = ROOT / "outputs" / "spc_runs" / SCENE_NAME / SPC_RUN
GS_DIR = ROOT / "outputs" / "gs_runs" / SCENE_NAME / GS_RUN
SCENES, MODELS, LOGS, FIGURES = (GS_DIR / d for d in ("scenes", "models", "logs", "figures"))
for folder in (SCENES, MODELS, LOGS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

manifest_path = SPC_DIR / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"No existe {manifest_path.relative_to(ROOT)}: ejecuta primero el Notebook 1 con RUN_NAME='{SPC_RUN}'.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
VIEWS = manifest["views"]
print(f"Corrida SPC: {SPC_RUN} ({len(VIEWS)} vistas, {len(manifest['conditions'])} condiciones) -> {GS_DIR.relative_to(ROOT)}")

## 3. Paso 3: preparar datos y poses

### Por qué se usan las poses del dataset

Cada escena combina las vistas reconstruidas con `cameras.bin`, `images.bin` y `points3D.bin` **originales**.
No se recalculan, por tres motivos:

1. **Aislamiento.** La comparación debe reflejar la calidad fotométrica de la adquisición. Si cada
   configuración tuviera poses propias, el error de calibración se mezclaría con el de reconstrucción.
2. **Referencia común.** Todas las corridas renderizan las vistas de test desde cámaras idénticas, así que
   se pueden medir contra la misma imagen original.
3. **Dependencias.** COLMAP no está instalado en este entorno. La re-estimación queda como trabajo futuro.

Esto es una suposición optimista, como se indica al inicio. Queda declarada en
`docs/00_plan_y_decisiones.md` y `docs/02_resultados.md`.

### Intrínsecos

`cameras.bin` no se reescribe. En Truck declara 1957×1091, y las imágenes son 979×546 ya en el dataset
original. El lector del repo calcula el FoV con el foco y el ancho de `cameras.bin`, sin mirar el archivo,
por lo que un tamaño distinto no altera la geometría mientras se conserve la razón de aspecto. Eso se
comprueba abajo. También se comprueba el punto principal, porque el repo lo asume centrado.

### Copia por escena

3DGS escribe `points3D.ply` dentro de `sparse/0/` la primera vez que abre una escena. Por eso cada escena
tiene su propia copia de `sparse/0`, y `data/` nunca se modifica. Las imágenes usan enlaces duros, sin costo
de disco.

In [ ]:
def link_or_copy(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def write_filtered_images_bin(src, dst, keep):
    """Reescribe images.bin solo con las vistas pedidas, conservando poses y observaciones sin cambios."""
    records = []
    with open(src, "rb") as handle:
        count = struct.unpack("<Q", handle.read(8))[0]
        for _ in range(count):
            header = handle.read(64)
            name = bytearray()
            while (char := handle.read(1)) != b"\x00":
                name += char
            raw_points = handle.read(8)
            n_points = struct.unpack("<Q", raw_points)[0]
            points = handle.read(24 * n_points)
            if Path(name.decode("utf-8")).name in keep:
                records.append(header + bytes(name) + b"\x00" + raw_points + points)
    with open(dst, "wb") as handle:
        handle.write(struct.pack("<Q", len(records)))
        handle.writelines(records)


def prepare_scene(name, images_dir):
    scene = SCENES / name
    images, sparse = scene / "images", scene / "sparse" / "0"
    images.mkdir(parents=True, exist_ok=True)
    sparse.mkdir(parents=True, exist_ok=True)
    for view in VIEWS:
        link_or_copy(images_dir / view, images / view)
    source = SCENE_DIR / "sparse" / "0"
    for file_name in ("cameras.bin", "points3D.bin"):
        if not (sparse / file_name).exists():
            shutil.copy2(source / file_name, sparse / file_name)
    if not (sparse / "images.bin").exists():
        if len(VIEWS) == len(list((SCENE_DIR / "images").iterdir())):
            shutil.copy2(source / "images.bin", sparse / "images.bin")
        else:
            write_filtered_images_bin(source / "images.bin", sparse / "images.bin", set(VIEWS))
    return scene


def check_scene(name, scene):
    """Sanity checks: fallan con un mensaje claro en lugar de dejar que train.py se detenga más tarde."""
    extr = read_extrinsics_binary(str(scene / "sparse" / "0" / "images.bin"))
    intr = read_intrinsics_binary(str(scene / "sparse" / "0" / "cameras.bin"))
    pose_names = sorted(Path(e.name).name for e in extr.values())
    files = sorted(p.name for p in (scene / "images").iterdir())
    if len(files) != len(pose_names):
        raise RuntimeError(f"{name}: {len(files)} imágenes para {len(pose_names)} poses.")
    if files != pose_names:
        difference = sorted(set(files) ^ set(pose_names))[:5]
        raise RuntimeError(f"{name}: los nombres de imágenes y poses no coinciden, por ejemplo {difference}.")
    sizes = set()
    for file_name in files:
        with Image.open(scene / "images" / file_name) as image:
            sizes.add(image.size)
    if len(sizes) != 1:
        raise RuntimeError(f"{name}: hay tamaños de imagen distintos: {sizes}.")
    width, height = sizes.pop()
    camera = intr[next(iter(extr.values())).camera_id]
    if camera.model not in ("PINHOLE", "SIMPLE_PINHOLE"):
        raise RuntimeError(f"{name}: modelo de cámara {camera.model} no soportado por el repo.")
    fx = camera.params[0]
    fy = camera.params[1] if camera.model == "PINHOLE" else fx
    cx, cy = camera.params[-2:]
    aspect_error = (width / height) / (camera.width / camera.height) - 1
    if abs(aspect_error) > 0.005:
        raise RuntimeError(f"{name}: el aspecto del archivo difiere {aspect_error:.2%} del calibrado.")
    center_error = max(abs(cx - camera.width / 2), abs(cy - camera.height / 2))
    if center_error > 1.0:
        raise RuntimeError(f"{name}: punto principal descentrado en {center_error:.2f} px.")
    scale = width / camera.width
    n_test = sum(1 for i in range(len(files)) if i % LLFF_HOLD == 0)
    return {
        "escena": name, "imagenes": len(files), "poses": len(pose_names), "train": len(files) - n_test, "test": n_test,
        "archivo": f"{width}x{height}", "calibrado": f"{camera.width}x{camera.height}", "error_aspecto_pct": 100 * aspect_error,
        "fx_a_escala": fx * scale, "fy_a_escala": fy * scale,
        "fov_x_grados": np.degrees(focal2fov(fx, camera.width)), "fov_y_grados": np.degrees(focal2fov(fy, camera.height)),
    }


conditions = {"baseline": {"pattern": "baseline", "ratio": 1.0, "method": "original", "images": SCENE_DIR / "images"}}
for entry in manifest["conditions"]:
    conditions[entry["name"]] = {
        "pattern": entry["pattern"], "ratio": entry["ratio"], "method": entry["method"],
        "images": ROOT / entry["images_dir"], "lam": entry.get("lam"),
    }

rows = []
for name, condition in conditions.items():
    rows.append(check_scene(name, prepare_scene(name, condition["images"])))
checks = pd.DataFrame(rows)
checks.to_csv(GS_DIR / "sanity_checks.csv", index=False)
display(checks.round(3))
print(f"Sanity checks OK en {len(checks)} escenas: imágenes = poses, nombres iguales, aspecto y punto principal correctos.")

## 4. Paso 4: entrenamiento

### Plan de corridas

Una corrida larga también produce el resultado de 7k: el entrenamiento guarda ese punto, y la tasa de
aprendizaje sigue el mismo calendario hasta 30k en todas las corridas (`position_lr_max_steps`). Por eso los
resultados a 7k son comparables entre corridas cortas y largas.

El orden sigue la prioridad acordada: baseline y Hadamard + TV primero, porque son la comparación central.

**Hiperparámetros.** Los del paper: pérdida L1 + D-SSIM con $\lambda = 0.2$ (§5.1, Ec. 7), densificación
cada 100 iteraciones desde la 500 hasta la 15k (§5.2) y SH de grado 3 (§7.1).

In [ ]:
def default_plan():
    plan = [["baseline", ITER_LONG]]
    ratios = sorted({c["ratio"] for c in conditions.values() if c["method"] == "tv"})
    plan += [[f"{HADAMARD}_r{round(r * 100):03d}_tv", ITER_LONG] for r in ratios]
    plan += [["control_resample", ITER_SHORT]]
    plan += [[f"{HADAMARD}_r{round(r * 100):03d}_direct", ITER_SHORT] for r in ratios]
    plan += [[f"random_r{round(r * 100):03d}_tv", ITER_SHORT] for r in ratios]
    return plan


plan = [list(item) for item in PLAN] if PLAN else default_plan()
unknown = [name for name, _ in plan if name not in conditions]
if unknown:
    raise ValueError(f"Corridas sin condición en el manifiesto: {unknown}")

run_config = {
    "dataset": DATASET, "spc_run": SPC_RUN, "fecha_inicio": time.strftime("%Y-%m-%d %H:%M:%S"),
    "resolution": RESOLUTION, "data_device": DATA_DEVICE, "densify_grad_threshold": DENSIFY_GRAD_THRESHOLD,
    "densify_until_iter": DENSIFY_UNTIL_ITER, "seed": SEED, "plan": plan,
}
config_path = GS_DIR / "run_config.json"
if not config_path.exists():
    config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
display(pd.DataFrame(plan, columns=["corrida", "iteraciones"]))


def gpu_temperature():
    output = subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
                            capture_output=True, text=True, check=True).stdout
    return float(output.strip().splitlines()[0])


def wait_for_cooldown():
    start = time.time()
    temperature = gpu_temperature()
    while temperature > MAX_START_TEMP_C and time.time() - start < COOLDOWN_MAX_S:
        time.sleep(15)
        temperature = gpu_temperature()
    return temperature, time.time() - start


def is_done(model, iterations):
    metrics_path, status_path = model / "metrics.json", model / "run_status.json"
    if not (metrics_path.exists() and status_path.exists()):
        return False
    status = json.loads(status_path.read_text(encoding="utf-8"))
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    return status.get("state") == "done" and metrics.get("iterations") == iterations and str(iterations) in metrics.get("eval", {})


def run_training(name, iterations):
    model, scene = MODELS / name, SCENES / name
    test_iters = [i for i in (500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000, 25000, 30000) if i < iterations] + [iterations]
    save_iters = sorted({i for i in (7000,) if i < iterations} | {iterations})
    checkpoints = [i for i in (3500, 7000, 15000, 22500) if i < iterations] + [iterations]
    command = [
        sys.executable, str(RUNNER), "-s", str(scene), "-m", str(model), "--eval",
        "--gt_images", str(SCENES / "baseline" / "images"), "-r", str(RESOLUTION),
        "--iterations", str(iterations), "--seed", str(SEED), "--data_device", DATA_DEVICE,
        "--densify_grad_threshold", str(DENSIFY_GRAD_THRESHOLD), "--densify_until_iter", str(DENSIFY_UNTIL_ITER),
        "--test_iterations", *map(str, test_iters), "--save_iterations", *map(str, save_iters),
        "--checkpoint_iterations", *map(str, checkpoints),
    ]
    environment = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONIOENCODING": "utf-8"}
    with open(LOGS / f"{name}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n{' '.join(command)}\n")
        log.flush()
        code = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, env=environment, cwd=GS_REPO).returncode
    if code == 0:  # conserva el checkpoint final, que permite extender la corrida
        for path in model.glob("chkpnt*.pth"):
            if path.name != f"chkpnt{iterations}.pth":
                path.unlink()
    return code


pending_oom = None
history = []
for name, iterations in plan:
    if ONLY is not None and name not in ONLY:
        continue
    if is_done(MODELS / name, iterations):
        print(f"[{name}] ya completa a {iterations} iteraciones; se omite.", flush=True)
        continue
    temperature, waited = wait_for_cooldown()
    print(f"[{name}] inicio {time.strftime('%H:%M:%S')}: {iterations} iteraciones, GPU {temperature:.0f} °C (espera {waited:.0f} s)", flush=True)
    start = time.time()
    code = run_training(name, iterations)
    state = {0: "ok", 3: "sin VRAM"}.get(code, f"error {code}")
    history.append({"corrida": name, "estado": state, "minutos": (time.time() - start) / 60})
    print(f"[{name}] {state} en {(time.time() - start) / 60:.1f} min. Log: {(LOGS / (name + '.log')).relative_to(ROOT)}", flush=True)
    if code == 3 and STOP_ON_OOM:
        pending_oom = name
        break

if pending_oom:
    raise RuntimeError(
        f"La corrida '{pending_oom}' agotó la VRAM. Para que las corridas sigan siendo comparables, cualquier ajuste "
        "debe aplicarse a todas. Prueba DATA_DEVICE='cpu' o un DENSIFY_GRAD_THRESHOLD mayor. Es posible reanudar: "
        "el progreso hasta el último checkpoint está guardado."
    )
failed = [h for h in history if h["estado"] != "ok"]
if failed:
    print("ATENCIÓN, corridas con error (revisa sus logs):", failed)

## 5. Comparación

Las métricas principales comparan el render contra la imagen **original** en las vistas de test. `metrics.py`
compararía contra el GT de la escena, que en una condición SPC ya está degradado: esas cifras se conservan
aparte, con el sufijo `_vs_input`, porque responden otra pregunta: qué tan bien 3DGS reproduce sus entradas.

In [ ]:
def collect_results():
    rows = []
    for name, condition in conditions.items():
        path = MODELS / name / "metrics.json"
        if not path.exists():
            continue
        metrics = json.loads(path.read_text(encoding="utf-8"))
        for iteration, values in metrics["eval"].items():
            rows.append({
                "corrida": name, "pattern": condition["pattern"], "ratio": condition["ratio"], "method": condition["method"],
                "iter": int(iteration), "iter_entrenadas": metrics["iterations"], **values,
                "train_min": (metrics.get("train_time_s") or 0) / 60,
                "vram_torch_pico_mib": metrics.get("torch_vram_peak_mib"), "vram_gpu_pico_mib": metrics.get("gpu_mem_peak_mib"),
                "temp_max_c": metrics.get("gpu_temp_max_c"), "temp_media_c": metrics.get("gpu_temp_mean_c"),
            })
    return pd.DataFrame(rows)


results = collect_results()
if results.empty:
    raise RuntimeError("Todavía no hay corridas terminadas.")
results.to_csv(GS_DIR / "resultados_3dgs.csv", index=False)
results.to_json(GS_DIR / "resultados_3dgs.json", orient="records", indent=2)
columns = ["corrida", "iter", "psnr", "ssim", "lpips", "psnr_vs_input", "n_gaussians", "ply_mb", "render_fps",
           "train_min", "vram_torch_pico_mib", "vram_gpu_pico_mib", "temp_max_c"]
display(results[columns].round(3).sort_values(["iter", "psnr"], ascending=[True, False]).reset_index(drop=True))

### Validación del baseline contra el paper

Antes de interpretar las corridas SPC, el baseline debe reproducir el paper (Tablas 7, 8 y 9: columna Truck de
*Ours-7k* y *Ours-30k*). Una diferencia grande indicaría un problema del entorno, y entonces las comparaciones
SPC no serían fiables. El límite de ±1 dB es un criterio de este proyecto, no del paper.

In [ ]:
PAPER_TRUCK = {7000: {"psnr": 23.506, "ssim": 0.840, "lpips": 0.209}, 30000: {"psnr": 25.187, "ssim": 0.879, "lpips": 0.148}}
BASELINE_TOLERANCE_DB = 1.0

rows = []
baseline = results[results["corrida"] == "baseline"]
for row in baseline.itertuples():
    if row.iter not in PAPER_TRUCK or row.iter_entrenadas not in PAPER_TRUCK or len(VIEWS) != len(list((SCENE_DIR / "images").iterdir())):
        continue
    if SCENE_NAME != "truck":
        continue
    paper = PAPER_TRUCK[row.iter]
    delta = row.psnr - paper["psnr"]
    rows.append({"iter": row.iter, "psnr": row.psnr, "psnr_paper": paper["psnr"], "delta_db": delta,
                 "ssim": row.ssim, "ssim_paper": paper["ssim"], "lpips": row.lpips, "lpips_paper": paper["lpips"],
                 "veredicto": "se acerca" if abs(delta) <= BASELINE_TOLERANCE_DB else "NO SE ACERCA"})
paper_check = pd.DataFrame(rows)
if paper_check.empty:
    print("Sin comparación con el paper: requiere Truck, todas las vistas y 7k o 30k iteraciones.")
else:
    paper_check.to_csv(GS_DIR / "baseline_vs_paper.csv", index=False)
    display(paper_check.round(3))
    if (paper_check["veredicto"] != "se acerca").any():
        print("ATENCIÓN: el baseline no se acerca al paper. No interpretar las corridas SPC hasta aclararlo.")

In [ ]:
# Paleta categórica validada (guía dataviz). El color sigue a la serie.
SERIES = {
    (HADAMARD, "tv"): ("Hadamard + TV", "#2a78d6"),
    (HADAMARD, "direct"): ("Hadamard directa", "#eb6834"),
    ("random", "tv"): ("Aleatorio + TV", "#1baf7a"),
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "axes.grid": True, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
    "figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight",
})
METRICS = [("psnr", "PSNR (dB), mayor es mejor"), ("ssim", "SSIM, mayor es mejor"), ("lpips", "LPIPS-VGG, menor es mejor")]


def legend_below(fig, axes):
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False, fontsize=9)
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))


def quality_figure(iteration):
    part = results[results["iter"] == iteration]
    if part[part["method"].isin(["tv", "direct"])].empty:
        return
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    for axis, (metric, title) in zip(axes, METRICS):
        for (pattern, method), (label, color) in SERIES.items():
            line = part[(part["pattern"] == pattern) & (part["method"] == method)].sort_values("ratio")
            if len(line):
                axis.plot(line["ratio"] * 100, line[metric], color=color, lw=2, marker="o", ms=6, label=label)
        for name, label, style in [("baseline", "Baseline (originales)", "-"), ("control_resample", "Control (remuestreo)", "--")]:
            line = part[part["corrida"] == name]
            if len(line):
                axis.axhline(line[metric].iloc[0], color=MUTED, lw=1.2, ls=style, label=label)
        axis.set_title(title, loc="left")
        axis.set_xlabel("Ratio de muestreo M/N (%)")
    fig.suptitle(f"Novel views de 3DGS a {iteration:,} iteraciones contra la imagen original, {SCENE_NAME}".replace(",", "."), x=0.01, ha="left")
    legend_below(fig, axes)
    fig.savefig(FIGURES / f"calidad_vs_ratio_{iteration}.png")
    plt.show()


for iteration in sorted(results["iter"].unique()):
    quality_figure(iteration)

### Curvas de entrenamiento

El PSNR de esta figura lo calcula `train.py` durante el entrenamiento y es **contra la entrada de cada escena**:
en una corrida SPC, contra la vista de test ya degradada. Sirve para ver la convergencia, no para comparar calidad.

In [ ]:
ITER_RE = re.compile(r"\[ITER (\d+)\] Evaluating test: L1 ([\d.eE+-]+) PSNR ([\d.eE+-]+)")


def test_curve(name):
    log = LOGS / f"{name}.log"
    curve = {}
    if log.exists():
        for match in ITER_RE.finditer(log.read_text(encoding="utf-8", errors="replace")):
            curve[int(match.group(1))] = float(match.group(3))  # al reanudar queda el último valor
    return sorted(curve.items())


def run_color(name):
    condition = conditions[name]
    return SERIES.get((condition["pattern"], condition["method"]), (name, MUTED))[1]


names = [n for n in results["corrida"].unique()]
long_runs = [n for n in names if n == "baseline" or conditions[n]["method"] == "tv" and conditions[n]["pattern"] == HADAMARD]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ramp = {0.05: "#a8cbf2", 0.10: "#6da5e6", 0.25: "#2a78d6", 0.50: "#134a8e"}
for name in long_runs:
    color = INK if name == "baseline" else ramp.get(conditions[name]["ratio"], "#2a78d6")
    label = "Baseline" if name == "baseline" else f"Hadamard + TV {conditions[name]['ratio']:.0%}"
    curve = test_curve(name)
    if curve:
        axes[0].plot(*zip(*curve), color=color, lw=2, marker="o", ms=4, label=label)
    log_path = MODELS / name / "train_log.csv"
    if log_path.exists():
        table = pd.read_csv(log_path)
        axes[1].plot(table["iteration"], table["loss"].rolling(10, min_periods=1).mean(), color=color, lw=1.5, label=label)
        axes[2].plot(table["iteration"], table["n_gaussians"] / 1e6, color=color, lw=2, label=label)
axes[0].set_title("PSNR en test contra la entrada (dB)", loc="left")
axes[1].set_title("Pérdida de entrenamiento, media móvil", loc="left")
axes[2].set_title("Gaussianas (millones)", loc="left")
for axis in axes:
    axis.set_xlabel("Iteración")
legend_below(fig, axes)
fig.savefig(FIGURES / "curvas_entrenamiento.png")
plt.show()

### Propagación del error: de la vista 2D al render 3D

Para cada configuración se comparan, en las **mismas vistas de test**:

- **2D:** la vista reconstruida por la SPC contra la original. 3DGS no entrena con estas vistas.
- **3D:** el render de 3DGS desde esa cámara, entrenado solo con las otras vistas.

Si 3D supera a 2D, la fusión multivista compensa parte del error de reconstrucción, aun tratándose de vistas
nuevas. Si queda por debajo, domina la pérdida propia de 3DGS. El baseline marca el techo de 3DGS en esta escena.

In [ ]:
summary_2d = pd.read_csv(SPC_DIR / "summary_2d.csv")
test_2d = summary_2d[summary_2d["split"] == "test"].rename(columns={"condicion": "corrida", "psnr": "psnr_2d", "ssim": "ssim_2d", "lpips": "lpips_2d"})
final = results[results["iter"] == results["iter_entrenadas"]]
# La comparación usa cada corrida en su última iteración; se reporta cuál es.
propagation = final.merge(test_2d[["corrida", "psnr_2d", "ssim_2d", "lpips_2d"]], on="corrida", how="inner")
base = final[final["corrida"] == "baseline"]
if len(base):
    propagation["psnr_baseline"] = base["psnr"].iloc[0]
    propagation["caida_vs_baseline_db"] = propagation["psnr_baseline"] - propagation["psnr"]
propagation["ganancia_3d_menos_2d_db"] = propagation["psnr"] - propagation["psnr_2d"]
columns = [c for c in ["corrida", "iter", "psnr_2d", "psnr", "ganancia_3d_menos_2d_db", "caida_vs_baseline_db", "ssim_2d", "ssim", "lpips_2d", "lpips"] if c in propagation]
propagation = propagation.sort_values(["pattern", "method", "ratio"])
propagation.to_csv(GS_DIR / "propagacion_2d_3d.csv", index=False)
display(propagation[columns].round(3).reset_index(drop=True))

series_part = propagation[propagation["method"].isin(["tv", "direct"])]
if len(series_part):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    for axis, (metric, title) in zip(axes, METRICS):
        for (pattern, method), (label, color) in SERIES.items():
            line = series_part[(series_part["pattern"] == pattern) & (series_part["method"] == method)].sort_values("ratio")
            if len(line):
                axis.plot(line["ratio"] * 100, line[f"{metric}_2d"], color=color, lw=1.5, ls="--", marker="o", ms=4, label=f"{label}, vista 2D")
                axis.plot(line["ratio"] * 100, line[metric], color=color, lw=2, marker="o", ms=6, label=f"{label}, render 3D")
        if len(base):
            axis.axhline(base[metric].iloc[0], color=MUTED, lw=1.2, label="Baseline 3DGS")
        axis.set_title(title, loc="left")
        axis.set_xlabel("Ratio de muestreo M/N (%)")
    legend_below(fig, axes)
    fig.savefig(FIGURES / "propagacion_2d_3d.png")
    plt.show()

### Renders: original, baseline y SPC

In [ ]:
def render_path(name, iteration, stem):
    return MODELS / name / "test_renders" / f"iteration_{iteration}" / f"{stem}.png"


stems = sorted({p.stem for p in (MODELS / "baseline" / "test_renders").glob("*/*.png")}) if (MODELS / "baseline").exists() else []
main_runs = [n for n in final.sort_values("ratio")["corrida"] if conditions[n]["pattern"] == HADAMARD and conditions[n]["method"] == "tv"]
if stems and main_runs:
    selected = [stems[len(stems) // 4], stems[(3 * len(stems)) // 4]]
    columns_runs = ["baseline", *main_runs]
    fig, axes = plt.subplots(len(selected), len(columns_runs) + 1, figsize=(3.2 * (len(columns_runs) + 1), 1.95 * len(selected)), squeeze=False)
    for row, stem in enumerate(selected):
        with Image.open(SCENES / "baseline" / "images" / f"{stem}.jpg") as image:
            axes[row, 0].imshow(image.convert("RGB"))
        axes[row, 0].set_title("Original" if row == 0 else "", fontsize=10)
        for col, name in enumerate(columns_runs, start=1):
            iteration = int(final[final["corrida"] == name]["iter"].iloc[0])
            path = render_path(name, iteration, stem)
            if path.exists():
                with Image.open(path) as image:
                    axes[row, col].imshow(image.convert("RGB"))
                values = pd.read_csv(MODELS / name / f"per_view_{iteration}.csv", dtype={"view": str}).set_index("view").loc[stem]
                label = "Baseline" if name == "baseline" else f"SPC {conditions[name]['ratio']:.0%}"
                axes[row, col].set_title(f"{label}: {values['psnr']:.1f} dB", fontsize=10)
        for axis in axes[row]:
            axis.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURES / "renders_gt_baseline_spc.png")
    plt.show()

### Exportar una animación

Recorre las cámaras de la captura, en orden, y genera un GIF lado a lado de baseline y SPC para las diapositivas.
Se ejecuta en el notebook, ya con todas las corridas terminadas, y libera la VRAM al final.

In [ ]:
@torch.no_grad()
def render_frames(name, iteration, n_frames, scale):
    from arguments import ModelParams, PipelineParams
    from gaussian_renderer import render
    from scene import GaussianModel, Scene

    parser = ArgumentParser()
    model_params, pipeline_params = ModelParams(parser), PipelineParams(parser)
    args = parser.parse_args(["-s", str(SCENES / name), "-m", str(MODELS / name), "--eval", "-r", str(RESOLUTION), "--data_device", "cpu"])
    gaussians = GaussianModel(args.sh_degree)
    scene = Scene(model_params.extract(args), gaussians, load_iteration=iteration, shuffle=False)
    cameras = sorted(scene.getTrainCameras() + scene.getTestCameras(), key=lambda c: c.image_name)
    picks = [cameras[i] for i in np.unique(np.linspace(0, len(cameras) - 1, n_frames).astype(int))]
    background = torch.zeros(3, device="cuda")
    pipe = pipeline_params.extract(args)
    frames = []
    for camera in picks:
        image = torch.clamp(render(camera, gaussians, pipe, background)["render"], 0, 1)
        array = (image.permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
        frame = Image.fromarray(array)
        frames.append(frame.resize((int(frame.width * scale), int(frame.height * scale)), Image.LANCZOS))
    del scene, gaussians
    torch.cuda.empty_cache()
    return frames


def export_side_by_side(spc_name, n_frames=48, scale=0.5, milliseconds=90):
    iteration = int(min(final[final["corrida"] == "baseline"]["iter"].iloc[0], final[final["corrida"] == spc_name]["iter"].iloc[0]))
    left, right = render_frames("baseline", iteration, n_frames, scale), render_frames(spc_name, iteration, n_frames, scale)
    frames = []
    for a, b in zip(left, right):
        canvas = Image.new("RGB", (a.width + b.width + 6, a.height + 22), (252, 252, 251))
        canvas.paste(a, (0, 22))
        canvas.paste(b, (a.width + 6, 22))
        draw = ImageDraw.Draw(canvas)
        draw.text((6, 5), f"Baseline, {iteration} iteraciones", fill=(11, 11, 11))
        draw.text((a.width + 12, 5), f"SPC: {spc_name}", fill=(11, 11, 11))
        frames.append(canvas.convert("P", palette=Image.ADAPTIVE, colors=200))
    target = FIGURES / f"recorrido_baseline_vs_{spc_name}.gif"
    frames[0].save(target, save_all=True, append_images=frames[1:], duration=milliseconds, loop=0, optimize=True)
    return target


if EXPORT_GIF and main_runs and "baseline" in set(final["corrida"]):
    middle = main_runs[len(main_runs) // 2]
    target = FIGURES / f"recorrido_baseline_vs_{middle}.gif"
    if target.exists():
        print("GIF ya existente:", target.relative_to(ROOT))
    else:
        target = export_side_by_side(middle)
        print(f"GIF: {target.relative_to(ROOT)} ({target.stat().st_size / 2**20:.1f} MB)")

In [ ]:
print("Resultados en", GS_DIR.relative_to(ROOT))
for name in ["sanity_checks.csv", "resultados_3dgs.csv", "resultados_3dgs.json", "propagacion_2d_3d.csv", "baseline_vs_paper.csv"]:
    print(" -", name, "OK" if (GS_DIR / name).exists() else "no generado")
print("Figuras:", sorted(p.name for p in FIGURES.iterdir()))
print("Corridas terminadas en esta sesión:", history if history else "ninguna; todo estaba completo")